## Download and Process the Wiktionary Dump

Library: https://github.com/tatuylonen/wikitextprocessor  
Dump: https://dumps.wikimedia.org/ukwiktionary/20260601/

In [1]:
import urllib.request

from functools import partial
from typing import Any

from wikitextprocessor import Wtp, WikiNode, NodeKind, Page
from wikitextprocessor.dumpparser import process_dump

In [2]:
urllib.request.urlretrieve(
    "https://dumps.wikimedia.org/ukwiktionary/20260601/ukwiktionary-20260601-pages-articles.xml.bz2",
    "ukwiktionary-20260601-pages-articles.xml.bz2"
)

('ukwiktionary-20260601-pages-articles.xml.bz2',
 <http.client.HTTPMessage at 0x108a65b90>)

In [3]:
wtp = Wtp(
    db_path="uk_20260601.db", lang_code="uk", project="wiktionary"
)

In [4]:
process_dump(
    wtp,
    "ukwiktionary-20260601-pages-articles.xml.bz2",
    {0, 10, 110, 828},
)

# 0: Main/Entries
# 10: Template
# 110: Thesaurus
# 828: Module (Lua scripts)

2026-06-24 11:55:57,929 INFO: skip_extract_dump: False, save_pages_path: None
2026-06-24 11:55:57,931 INFO: dump file path: ukwiktionary-20260601-pages-articles.xml.bz2
2026-06-24 11:55:58,558 INFO:   ... 10000 raw pages collected
2026-06-24 11:55:58,987 INFO:   ... 20000 raw pages collected
2026-06-24 11:55:59,383 INFO:   ... 30000 raw pages collected
2026-06-24 11:55:59,755 INFO:   ... 40000 raw pages collected
2026-06-24 11:56:00,331 INFO:   ... 50000 raw pages collected
2026-06-24 11:56:00,819 INFO:   ... 60000 raw pages collected
2026-06-24 11:56:01,319 INFO:   ... 70000 raw pages collected


### Page Example

In [5]:
page = wtp.get_page("кіт", 0)
wtp.start_page(page.title)
print(page.body)

{{Cf|Кіт}}

{{=uk=}}
===Морфосинтаксичні ознаки===
{{імен uk 1*b m a|склади=кі́т|кі́т|кот}}

{{морфо-uk|кіт}}
=== Вимова ===
* {{transcriptions-uk|кі́т|коти́}}
* {{audio|Uk-кіт.ogg|прослухати вимову}}
* {{транскрипція|к’іт}}
===Семантичні властивості===
{{іл|Cat03.jpg|кіт}}
{{іл|Kop van een kat PK-T-AW-5041, PK-T-AW-5040.tiff}}
==== Значення ====
# {{зоол.|uk}} cвійська [[тварина]] родини котячих, що знищує [[миша|мишей]] і [[щур]]ів; [[самець]] [[кішка|кішки]] {{семантика|синоніми=|антоніми=|гіпероніми=ссавець, тварина|гіпоніми=}} {{приклад2|{{виділ|Кіт}} [[Люцифер]] був, як і його господар, із [[примха]]ми. |джерело=http://zhogoli.com.ua/images/zhurnal/Svitlo17.pdf}}

{{списки семантичних зв’язків}}

=== Усталені словосполучення, фразеологізми ===
* [[кіт у мішку]]
* [[кіт свійський]]
===Споріднені слова===
{{спорідн|
|згруб= котисько, котище, котяра
|змп= котик, киця, кішечка, кішка, кошеня, кошенятко, кошеняточко
|власні назви= 
|прізвища= 
|топоніми= 
|іменники= котятина 
|прикмет

## Extract UA-RU

In [6]:
import re

In [7]:
UK_PAGE_RE = re.compile(r"\{(-uk-|=uk=)[}|]")

RU_TRANSLATION_RE = re.compile(r"ru=(.+?)\n")

In [8]:
ua_words = dict()

for page in wtp.get_all_pages([0]):
    if not page.body:
        continue
    # only Ukrainian pages with one-word entries
    if UK_PAGE_RE.search(page.body) and " " not in page.title:
        uk_word = page.title
        ru_words = set(
            m.group(1).strip()
            for m in RU_TRANSLATION_RE.finditer(page.body)
        )
        ua_words[uk_word] = ru_words

In [9]:
# one-word UA vs one-word non-UA
print(len(ua_words.items()), len(list(wtp.get_all_pages([0]))))

39031 65131


In [10]:
def normalize(ru_string):
    """Extract the word from the Wiktionary markup; sometimes broken."""
    # Clean-up
    ru_string = ru_string.rstrip(",")
    # романтик {{m}}
    ru_string = re.sub(r" \{\{(?:[mfn]|pl?|мн\.)\|?\}\}", "", ru_string)
    # {{позначка|(на металі)}} {{t|ru|щербина|f}}
    ru_string = re.sub(r"\{\{[^}]+\}\} ", "", ru_string)
    # ''[[комп'ютер|комп.]]'' [[скачать]]
    ru_string = re.sub(r"''\[\[[^\]]+\]\]'' ", "", ru_string)
    # (мн. ч.) гу́ли
    ru_string = re.sub(r"\(мн\. ч\.\) ", "", ru_string)
    
    # Extract
    # [[:ru:нас|нас]]
    ru_string = re.sub(r"\[\[:ru:[^|]+\|(.+)\]\]", r"\1", ru_string)
    # [[дилетант#|дилетант]] {{m}}
    ru_string = re.sub(r"\[\[(?:[^\]]+#(?:Російська)?\|)?([^\]]+)\]\]", r"\1", ru_string)
    # абаз{{wiktionary|ru|абаз}}, [[политтехнолог]] {{wiktionary|ru|политтехнолог}}
    ru_string = re.sub(r"([^{\[ ]+|\[\[[^{\[ ]+\]\]) ?\{\{wiktionary\|ru\|.*\}\}", r"\1", ru_string)
    # {{t|подраздел|m}}, {{t|органайзер|m}}
    ru_string = re.sub(r"\{\{t[o+]?\|(?:ru?|pl)?\|?([^|(]+)(?: \(.*\))?(?:\|.*)?\}\}", r"\1", ru_string)
    # {{t|ru|высота|f}}, {{t|ru|Купчин|}}, {{t|ru|этот|m|s|tr=е́тот}}, {{t|ru|кадило (растение)|n}}, {{ru|бригада|f}}
    ru_string = re.sub(r"\{\{(?:t?[o+]?\|)?(?:ru?|pl)?\|([^|(]+)(?: \(.*\))?(?:\|.*)?\}\}", r"\1", ru_string)
    
    # Normalize
    ru_string = re.sub("а́", "а", ru_string)
    ru_string = re.sub("е́", "е", ru_string)
    ru_string = re.sub("о́", "о", ru_string)
    ru_string = re.sub("у́", "у", ru_string)
    ru_string = re.sub("и́", "и", ru_string)
    ru_string = re.sub("ы́", "ы", ru_string)
    ru_string = re.sub("я́", "я", ru_string)
    ru_string = re.sub("ю́", "ю", ru_string)

    # More clean-up
    ru_string = ru_string.rstrip("}]")
    ru_string = ru_string.lstrip("{[")

    return ru_string

In [11]:
processed = dict()

for k, v in ua_words.items():
    new_v = set()
    for val in v:
        # Clean-up
        val = re.sub("&nbsp;", " ", val)
        # |sk=|sw=...
        val = re.sub(r"\|(sk|rw)=(\|[^=}]+=)*", "", val)
        # <!-- мертвая вода: антисептик, дезинфектант -->
        val = re.sub(r" ?<!--[^>]+-->", "", val)
        # {{позначка|перен.}}, {{позначка|бран.}} {{t|ru|дрянь|f}}
        val = re.sub(r"\{\{позначка\|[^}]+\}\}(?:, \{\{позначка\|[^}]+\}\})? ", "", val)
        # {{t|ru|конфере́нция|f}}, {{t|ru|съезд|m}} {{qualifier|congress, convention}}
        val = re.sub(r" \{\{(?:q(?:ualifier)?|позначка)\|[^}]+\}\}", "", val)
        
        if val in {"", "}}", "{{t|ru||}}"}:
            continue
        
        for v_val in re.split(r" *[,;] +", val):
            if "PAGENAME" in v_val:
                # the translation is identical to the Ukrainian word
                new_v.add(k)
            else:
                new_v.add(normalize(v_val))
                
    processed[k] = list(new_v)

In [12]:
ua_words["яловичина"]

{'{{t+|ru|говя́дина|f}}, {{t+|ru|теля́тина|f}} {{qualifier|veal}}'}

In [13]:
processed["яловичина"]

['говядина', 'телятина']

### Dump Translations into a File

In [14]:
import json

In [15]:
non_empty = {k: v for k, v in processed.items() if v and k[0].islower() and k[-1] != "-"}
with open("wiktionary-ua2ru.json", "w", encoding="utf-8") as f:
    json.dump(dict(sorted(non_empty.items())), f, ensure_ascii=False, indent=4)

In [16]:
len(non_empty)

6561